In [3]:
import numpy as np
import pennylane as qml
import itertools
from itertools import combinations
from scipy.optimize import minimize
from math import comb

In [4]:
def make_dicke_state(n, k):
        if k < 0 or k > n:
            raise ValueError("k must satisfy 0 <= k <= n")
    
        state = np.zeros(2**n, dtype=complex)
        weight_k_states = list(combinations(range(n), k))
        
        amplitude = 1 / np.sqrt(len(weight_k_states))

        #convert bit string to integer
        for one_positions in weight_k_states:
            basis_index=0
            for wire in one_positions:
                basis_index+=2**(n-1-wire)
            state[basis_index] = amplitude

        

        d_state = np.array(state, requires_grad=False)
        if (comb(n,k)!=np.count_nonzero(d_state)):
            raise ValueError("amplitude count incorrect")
    
        return d_state

In [5]:
def clique_expansion(hyperedges):
    mixer_edges = set()

    for edge in hyperedges:
        for u, v in itertools.combinations(edge, 2):
            mixer_edges.add(tuple(sorted((u, v))))

    return sorted(mixer_edges)

In [9]:
def build_cost_terms(hyperedge_list):
    """
    Returns a list of expanded even-order Z terms.

    Each entry:
        (hyperedge_idx, wires, coefficient) where wires is a tuple such as (0, 2) or (0, 1, 2, 3).
    """
    cost_terms = []

    for edge_idx, edge in enumerate(hyperedge_list):
        edge = tuple(edge)
        k = len(edge)

        # All nonconstant terms have the same coefficient
        coeff = -1 / (2 ** (k - 1))

        # Keep only even-order Z products: 2, 4, 6, ...
        for order in range(2, k + 1, 2):
            for wires in combinations(edge, order):
                cost_terms.append(
                    (edge_idx, wires, coeff)
                )

    return cost_terms

In [6]:
def mincut_balanced(n:int,
                    hyperedge_list,
                    seed: int,
                    num_layers: int,
                    scaling_constant: float,
                    step_size: float, 
                    max_trials: int,):
    edges = clique_expansion(hyperedge_list)
    num_edges = len(edges)
    num_terms = num_cost_terms(hyperedge_list)

    # Mixer interactions
    edges = clique_expansion(hyperedge_list)
    num_edges = len(edges)

    # Exact expanded cost terms
    cost_terms = build_cost_terms(hyperedge_list)
    num_cost_terms = len(cost_terms)

    # get cost/mixer parameters
    beta_params = scaling_constant * rng.random([num_edges, num_layers],requires_grad=True)
    gamma_params = scaling_constant * rng.random([num_cost_terms, num_layers],requires_grad=True)

In [ ]:
|